# Buck Converter — Interactive Explorer (rev 1)

A synchronous **buck converter** steps $V_{in}$ down to $V_{out}$ through a switched LC filter and
regulates it with a feedback loop. Its behaviour splits along two axes that this explorer exposes as
toggles: **conduction mode** (CCM vs DCM — does the inductor current reach zero each cycle?) and
**control scheme** (voltage mode vs peak current mode). Each of the four combinations has a
*different* small-signal plant, so each needs a different compensator — the classic buck loop-design
problem.

**Key relations**

$$M=\frac{V_{out}}{V_{in}},\qquad K=\frac{2\,L\,f_{sw}}{R_{load}},\qquad
\text{CCM iff } K > 1-M,\qquad D_{CCM}=M,\qquad D_{DCM}=M\sqrt{\frac{K}{1-M}}$$

$$\text{VM-CCM:}\quad G_p(s)=\frac{V_{in}}{V_{osc}}\cdot
\frac{1+s\,r_C C}{1+s/(Q\,\omega_0)+(s/\omega_0)^2},\qquad
f_0=\frac{1}{2\pi\sqrt{LC}},\qquad f_{ESR}=\frac{1}{2\pi\,r_C C}$$

$$\text{CM-CCM (Ridley):}\quad G_p(s)=\frac{R_{load}}{R_i}\,K_x\,
\frac{1+s\,r_C C}{(1+s/\omega_p)\,\big(1+\frac{s}{\omega_n Q_p}+\frac{s^2}{\omega_n^2}\big)},\qquad
\omega_n=\pi f_{sw},\qquad Q_p=\frac{1}{\pi\,(m_c(1{-}D)-\tfrac12)}$$

VM-DCM collapses to a dominant single pole $\omega_p=\frac{2-M}{(1-M)R_{load}C}$; CM-DCM is
first-order too. The loop is $T(s)=G_p\,G_c$ with an auto-designed **Type III** (VM-CCM) or
**Type II** compensator placed by the AN-1162 rules and gain-normalized so $|T(f_{c,t})|=1$.
$Q_p<0$ (i.e. $m_c(1{-}D)<\tfrac12$, where $m_c=1+S_e/S_n$) is the **subharmonic instability** —
peak current mode with $D>0.5$ and insufficient slope compensation $S_e$.

**Validated default design.** The sliders start on the **AN-1162 Type III-B worked example**
(International Rectifier / Infineon, IR3842 SupIRBuck) [**[1]**](#References): $V_{in}=12$ V,
$V_{out}=1.8$ V, $I_o=4$ A ($R_{load}=0.45\,\Omega$), $L=1.5$ µH, $C_o=4\times10.8$ µF effective
MLCC (3 mΩ each $\Rightarrow r_C=0.75$ mΩ), $f_{sw}=600$ kHz, $V_{osc}=1.8$ V,
target $F_0=100$ kHz. This single source supplies every default except $r_L$ (5 mΩ assumed
typical DCR; [1] keeps $R_L$ symbolic and gives no value) and the current-mode extras
$R_i$, $S_e$ (from the supporting Ridley/Richtek models [**[3]**](#References),
[**[4]**](#References)). The model reproduces AN-1162's measured loop — see the validation
section below the controls.

### Layout
- **Schematic + sliders + toggles on the left, formula ledger on the right** of both.
- **Row 1:** load current step $I_{load}(t)$ ($+50\%$) and closed-loop output deviation
  $\Delta V_{out}(t)$ — x-axes aligned in time.
- **Row 2:** steady-state switching waveforms over 3 cycles — inductor current $i_L(t)$
  (triangle in CCM, touches zero in DCM) and output ripple, x-aligned.
- **Row 3:** **open-loop** decomposed into the plant $G_p$ (black) and the inverse compensator
  $1/G_c$ (gray) overlaid — their intercept is the loop crossover; and the **loop gain** $|T|$
  (20 dB primary grid).
- **Row 4:** the matching **phases** $\angle G_p$, $\angle(1/G_c){-}180°$ overlaid (gap at
  crossover = PM), and $\angle T$ (90° primary grid).
- **Formula ledger**: each row is `formula  =  value` (solid underline) `=  previous value`
  (dashed underline), all in black.

### Behaviour
- Each parameter has a **slider + type-in box**, centred on the AN-1162 value and spanning
  **0.01×–100×** ($S_e$ is linear so 0 = no slope compensation is reachable). The **▲/▼ buttons**
  step in half-decade multiples of the original value (0.01×, 0.032×, 0.1×, 0.32×, 1×, 3.2×, 10×,
  32×, 100×); the $S_e$ buttons step ±0.34 V/µs (half of the default on-slope $S_n$).
- **mode** (auto/CCM/DCM), **control** (voltage/current), **comp** (auto/type2/type3) toggles
  switch the small-signal model; *auto* detects mode from $K \gtrless 1-M$ and picks the
  compensator per AN-1162 Table 1. A warning banner flags forced-mode disagreement, subharmonic
  risk, and $V_{out}\ge V_{in}$.
- Plot traces are **black**; on replot the previous curve becomes a **dashed/dotted black ghost
  that fades**, gone by the 3rd replot. **Axes only grow.** **Reset** clears everything.
- Load step starts at $t=-10\%$ of full scale. Use the toolbar box-zoom (drag & release) to zoom
  transients.
- **Mouse move** → vertical cursor + value label, mirrored into sibling plots (step↔step,
  ripple↔ripple, freq↔freq); labels read both the current and the previous (dashed) curve. On the
  open-loop pair the cursor also shows the **$G_p\!\cdot\!G_c$ gap arrow** (loop gain in dB +
  slope in dB/dec; phase gap in ° = PM at crossover).
- **Click** → sticky cursor that fades on the next click elsewhere and is removed on a third.
- **Difference shading** between newest and previous curve on **every** plot: **green** where
  higher, **red** where lower.
- **Color** button (right of Reset): 1st click tints every component its own colour in *both* the
  schematic and the formulas (e.g. $L$ orange everywhere it appears, including inside $f_0$); each
  further click highlights one component at a time; **Reset** clears the colour. Clicking a
  **slider's name** highlights just that component (click again to clear); adjusting a slider also
  highlights its component.

> Run all cells. Needs `ipywidgets` + `ipympl` (`%matplotlib widget`) for live mouse interaction.

## Design spec vs. model result (defaults)

Requirements are the AN-1162 Type III-B design targets [**[1]**](#References); results are this
notebook's model at the default slider values.

| Spec | Requirement [1] | Model result | Meets? |
|---|---|---|---|
| Output voltage | 1.8 V from 12 V | $D=0.152$ regulated | ✓ |
| Max load current | 4 A ($R_{load}=0.45\,\Omega$) | 4 A operating point, CCM ($K=4.0 > 1{-}M=0.85$) | ✓ |
| Switching frequency | 600 kHz | 600 kHz | ✓ |
| Inductor ripple $\Delta I_L$ | ≈ 40 % of $I_o$ (power-stage guideline, eq. A1) | 1.70 A = 42 % | ✓ |
| Output ripple $\Delta V_{out}$ | small vs. transient budget (MLCC) | ≈ 9.4 mV = 0.5 % | ✓ |
| Loop crossover $F_0$ | $f_{sw}/6$ = 100 kHz (within the 1/10–1/5 rule) | 100.0 kHz | ✓ |
| Phase margin | > 45° | 52.9° | ✓ |
| Gain margin | (not specified; >10 dB typical) | 19.5 dB | ✓ |

In [ ]:
%matplotlib widget
%config InlineBackend.figure_formats = ['svg']   # crisp vector output where inline is used
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'            # keep text selectable in SVG
plt.rcParams['figure.max_open_warning'] = 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy import signal
from collections import deque

VOSC = 1.8   # PWM ramp amplitude (V), fixed from AN-1162 [1] (IR3842 design)


def _eng(x, unit):
    if x is None or not np.isfinite(x):
        return f"-- {unit}"
    if x == 0:
        return f"0 {unit}"
    for s, p in [(1e9,'G'),(1e6,'M'),(1e3,'k'),(1,''),(1e-3,'m'),
                 (1e-6,'u'),(1e-9,'n'),(1e-12,'p'),(1e-15,'f')]:
        if abs(x) >= s:
            return f"{x/s:.3g} {p}{unit}"
    return f"{x:.3g} {unit}"


def _fnum(v, fmt='{:.3f}', suffix=''):
    return ('--' if v is None or not np.isfinite(v) else fmt.format(v)) + suffix


# ================================================================ Module 1: DC
def solve_dc(P):
    """Steady-state operating point.  Conduction parameter K = 2 L fsw / Rload;
       CCM iff K > 1 - M (ideal D = M).  DCM: D = M sqrt(K/(1-M)), which inverts
       M = 2/(1+sqrt(1+4K/D^2))  (Erickson & Maksimovic [2])."""
    Vin, Vout = P['Vin'], P['Vout']
    L, fsw, R = P['L'], P['fsw'], P['Rload']
    warn = []
    M = Vout/Vin
    if M >= 0.95:
        warn.append(f"V_out/V_in = {M:.2f} &ge; 0.95 — not a buck operating point; clamped to M = 0.95")
        M = 0.95
        Vout = M*Vin
    K = 2.0*L*fsw/R
    Kcrit = 1.0 - M
    det = 'CCM' if K > Kcrit else 'DCM'
    mode = P.get('mode', 'auto')
    if mode == 'auto':
        mode = det
    elif mode != det:
        warn.append(f"forced {mode}, but K = {K:.3g} vs 1-M = {Kcrit:.3g} says {det} — analyzing off the natural operating point")
    Io = Vout/R
    if mode == 'CCM':
        D = min(0.98, (Vout + Io*P['rL'])/Vin)   # ideal D=M, refined with DCR drop
        dIL = Vout*(1.0-D)/(L*fsw)
        D2 = 1.0 - D
        ipk = Io + dIL/2.0
    else:
        D = M*np.sqrt(K/(1.0-M))
        dIL = (Vin - Vout)*D/(L*fsw)             # = peak current in DCM
        ipk = dIL
        D2 = D*(Vin - Vout)/Vout                 # demagnetisation interval
    dVout = dIL*(P['rC'] + 1.0/(8.0*P['C']*fsw))
    return dict(M=M, K=K, Kcrit=Kcrit, mode=mode, detected=det, D=D, D2=D2,
                Io=Io, dIL=dIL, ipk=ipk, dVout=dVout, Vout=Vout, warn=warn)


# ============================================================= Module 2: plant
def plant(P, dc):
    """Small-signal control-to-output as (num, den) polynomials in s.
       Voltage mode: d-hat -> vout including the 1/Vosc PWM gain (AN-1162 [1]).
       Current mode: vc-hat -> vout — Ridley CCM model with the fsw/2 sampling
       double pole [3][4]; first-order approximation in DCM [2]."""
    Vin, Vout = P['Vin'], dc['Vout']
    L, C, rC, rL = P['L'], P['C'], P['rC'], P['rL']
    R, fsw = P['Rload'], P['fsw']
    M, D = dc['M'], dc['D']
    warn = []
    info = {}
    if P['control'] == 'voltage':
        if dc['mode'] == 'CCM':
            k0 = (Vin/VOSC)*R/(R+rL)
            a1 = (L + C*(rC*R + rC*rL + rL*R))/(R+rL)
            a2 = L*C*(R+rC)/(R+rL)
            num = k0*np.array([rC*C, 1.0])
            den = np.array([a2, a1, 1.0])
            info['f0'] = 1.0/(2*np.pi*np.sqrt(a2))
            info['Q'] = np.sqrt(a2)/a1
            info['fp'] = np.nan
        else:
            Gd0 = (2.0*Vout/D)*(1.0-M)/(2.0-M)/VOSC
            wp = (2.0-M)/((1.0-M)*R*C)
            wp2 = 2.0*fsw                          # HF pole near fsw/pi -> 2*fsw rad/s
            num = Gd0*np.array([rC*C, 1.0])
            den = np.polymul([1.0/wp, 1.0], [1.0/wp2, 1.0])
            info['fp'] = wp/(2*np.pi)
    else:                                          # peak current-mode control
        Ri = P['Ri']
        Dp = 1.0 - D
        Sn = Ri*(Vin - Vout)/L                     # sensed on-slope (V/s)
        mc = 1.0 + (P['Se']/Sn if Sn > 0 else 0.0)
        Ts = 1.0/fsw
        info['mc'] = mc
        info['Sn'] = Sn
        if dc['mode'] == 'CCM':
            kq = mc*Dp - 0.5
            Kx = 1.0/(1.0 + R*Ts*kq/L)
            wp = (1.0 + R*Ts*kq/L)/(R*C)
            wn = np.pi*fsw
            Qp = 1.0/(np.pi*kq) if abs(kq) > 1e-12 else np.inf
            info['Qp'] = Qp
            info['fp'] = wp/(2*np.pi)
            if kq <= 0:
                warn.append(f"subharmonic: m_c(1-D)-&frac12; = {kq:.3f} &le; 0 (D = {D:.2f}) — increase S_e")
            num = (R/Ri)*Kx*np.array([rC*C, 1.0])
            den = np.polymul([1.0/wp, 1.0], [1.0/wn**2, 1.0/(wn*Qp), 1.0])
        else:
            Kdc = (R/Ri)*(1.0-M)/(2.0-M)
            wp = (2.0-M)/((1.0-M)*R*C)
            num = Kdc*np.array([rC*C, 1.0])
            den = np.array([1.0/wp, 1.0])
            info['fp'] = wp/(2*np.pi)
    info['fesr'] = 1.0/(2*np.pi*rC*C) if rC > 0 else np.inf
    return num, den, info, warn


# ======================================================== Module 3: compensator
def compensator(P, dc, pinfo):
    """Auto-placed Type II / Type III following AN-1162 [1]:
       III-B (fESR > fsw/2, MLCC): 70-deg lead pair straddling fc,t, fz1 = fz2/2, fp3 = fsw/2.
       III-A (fESR < fsw/2): fz2 = fLC, fz1 = 0.75 fLC, fp2 = fESR, fp3 = fsw/2.
       Type II: zero cancels the dominant plant pole (0.75 fLC for VM-CCM),
       pole at min(fESR, fsw/2) [1][4].  Integrator gain set numerically at fc,t."""
    fsw, fct = P['fsw'], P['fct']
    kind = P.get('comp', 'auto')
    fesr = pinfo['fesr']
    if kind == 'auto':
        kind = 'type3' if (P['control'] == 'voltage' and dc['mode'] == 'CCM') else 'type2'
    if kind == 'type3':
        f0 = pinfo.get('f0')
        if f0 is None or not np.isfinite(f0):
            f0 = pinfo.get('fp', fct/5)
        if fesr > fsw/2:
            th = np.radians(70.0)
            fz2 = fct*np.sqrt((1-np.sin(th))/(1+np.sin(th)))
            fp2 = fct*np.sqrt((1+np.sin(th))/(1-np.sin(th)))
            fz1 = 0.5*fz2
        else:
            fz2 = f0
            fz1 = 0.75*f0
            fp2 = fesr
        fp3 = fsw/2.0
        zs, ps = [fz1, fz2], [fp2, fp3]
        num = np.polymul([1.0/(2*np.pi*fz1), 1.0], [1.0/(2*np.pi*fz2), 1.0])
        den = np.polymul([1.0, 0.0], np.polymul([1.0/(2*np.pi*fp2), 1.0],
                                                [1.0/(2*np.pi*fp3), 1.0]))
    else:
        if P['control'] == 'voltage' and dc['mode'] == 'CCM':
            fz = 0.75*pinfo.get('f0', fct/5)
        else:
            fz = pinfo.get('fp', fct/5)
        if not np.isfinite(fz) or fz <= 0:
            fz = fct/5
        fpc = min(fesr, fsw/2.0)
        zs, ps = [fz, np.nan], [fpc, np.nan]
        num = np.array([1.0/(2*np.pi*fz), 1.0])
        den = np.polymul([1.0, 0.0], [1.0/(2*np.pi*fpc), 1.0])
    return dict(kind=kind, num=num, den=den, zs=zs, ps=ps)


def loop_polys(P, dc):
    """T(s) = Gplant * Gc as polynomial ratio; Gc gain normalized so |T(fc,t)| = 1."""
    pn, pd, pinfo, w1 = plant(P, dc)
    comp = compensator(P, dc, pinfo)
    s0 = 2j*np.pi*P['fct']
    g1 = abs(np.polyval(pn, s0)/np.polyval(pd, s0) *
             np.polyval(comp['num'], s0)/np.polyval(comp['den'], s0))
    KI = 1.0/g1 if g1 > 0 else 1.0
    cn = KI*comp['num']
    return dict(pn=pn, pd=pd, cn=cn, cd=comp['den'],
                Tn=np.polymul(pn, cn), Td=np.polymul(pd, comp['den']),
                pinfo=pinfo, comp=comp, KI=KI, warn=w1)


# ============================================================ Module 4: analyze
def analyze(lp, fsw, n=6000):
    """Crossover, phase margin, gain margin from the loop-gain polynomials."""
    f = np.logspace(0, np.log10(5*fsw), n)
    s = 2j*np.pi*f
    T = np.polyval(lp['Tn'], s)/np.polyval(lp['Td'], s)
    mag = 20*np.log10(np.abs(T))
    ph = np.degrees(np.unwrap(np.angle(T)))
    idx = np.where((mag[:-1] > 0) & (mag[1:] <= 0))[0]
    if idx.size:
        i = idx[0]
        fc = np.interp(0.0, [mag[i+1], mag[i]], [f[i+1], f[i]])
        pm = 180.0 + np.interp(np.log10(fc), np.log10(f), ph)
    else:
        fc, pm = np.nan, np.nan
    gm = np.nan
    idx = np.where((ph[:-1] > -180) & (ph[1:] <= -180))[0]
    if idx.size:
        i = idx[0]
        f180 = np.interp(-180.0, [ph[i+1], ph[i]], [f[i+1], f[i]])
        gm = -np.interp(np.log10(f180), np.log10(f), mag)
    return dict(fc=fc, pm=pm, gm=gm)


def responses(P, dc, lp, n=2000):
    """Bode arrays: plant Gp, inverse compensator 1/Gc (shifted phase), loop T.
       Grid spans 0.1x lowest corner to 10x highest (fsw)."""
    corners = [c for c in ([lp['pinfo'].get('f0'), lp['pinfo'].get('fp'),
                            lp['pinfo'].get('fesr'), P['fct'], P['fsw']] +
                           list(lp['comp']['zs']) + list(lp['comp']['ps']))
               if c is not None and np.isfinite(c) and c > 0]
    lo, hi = 0.1*min(corners), 10.0*max(min(c, 10*P['fsw']) for c in corners)
    f = np.logspace(np.log10(lo), np.log10(hi), n)
    s = 2j*np.pi*f
    Gp = np.polyval(lp['pn'], s)/np.polyval(lp['pd'], s)
    Gc = np.polyval(lp['cn'], s)/np.polyval(lp['cd'], s)
    T = Gp*Gc
    deg = lambda z: np.degrees(np.unwrap(np.angle(z)))
    invGc_ph = deg(1.0/Gc)
    return dict(f=f,
                Gp_mag=20*np.log10(np.abs(Gp)),      Gp_ph=deg(Gp),
                invGc_mag=-20*np.log10(np.abs(Gc)),  invGc_ph_s=invGc_ph - 180.0,
                T_mag=20*np.log10(np.abs(T)),        T_ph=deg(T))


# ====================================================== Module 5: time domain
def ripple_waves(P, dc, ncyc=3, N=1800):
    """Steady-state switching waveforms over ncyc cycles: inductor current and
       output ripple v_out(AC) = v_C(AC) + rC*(iL - Io)."""
    Ts = 1.0/P['fsw']
    t = np.linspace(0.0, ncyc*Ts, N)
    tm = np.mod(t, Ts)
    Vin, Vout = P['Vin'], dc['Vout']
    L, C, rC = P['L'], P['C'], P['rC']
    D, D2, Io = dc['D'], dc['D2'], dc['Io']
    s1, s2 = (Vin - Vout)/L, -Vout/L
    if dc['mode'] == 'CCM':
        i0 = Io - dc['dIL']/2.0
        iL = np.where(tm < D*Ts, i0 + s1*tm, i0 + s1*D*Ts + s2*(tm - D*Ts))
    else:
        iL = np.zeros_like(tm)
        m1 = tm < D*Ts
        m2 = (tm >= D*Ts) & (tm < (D+D2)*Ts)
        iL[m1] = s1*tm[m1]
        iL[m2] = dc['ipk'] + s2*(tm[m2] - D*Ts)
        iL = np.maximum(iL, 0.0)
    ic = iL - np.mean(iL)
    vC = np.concatenate([[0.0], np.cumsum(0.5*(ic[1:]+ic[:-1])*np.diff(t))])/C
    vout_ac = (vC - np.mean(vC)) + rC*ic
    return t, iL, vout_ac


def load_step(P, dc, lp, N=4000):
    """Closed-loop load-step: dVout(t) = -dI * step of Zout,cl, with
       Zout,cl = Zout,ol / (1 + T) and Zout,ol = (rL+sL) || (rC+1/sC) || Rload."""
    L, C, rC, rL, R = P['L'], P['C'], P['rC'], P['rL'], P['Rload']
    z1n, z1d = np.array([L, rL]), np.array([1.0])
    z2n, z2d = np.array([rC*C, 1.0]), np.array([C, 0.0])
    z3n, z3d = np.array([R]), np.array([1.0])
    Yn = np.polyadd(np.polyadd(np.polymul(z1d, np.polymul(z2n, z3n)),
                               np.polymul(z2d, np.polymul(z1n, z3n))),
                    np.polymul(z3d, np.polymul(z1n, z2n)))
    Zd_ol = Yn
    Zn_ol = np.polymul(z1n, np.polymul(z2n, z3n))
    num = np.polymul(Zn_ol, lp['Td'])
    den = np.polymul(Zd_ol, np.polyadd(lp['Td'], lp['Tn']))
    dI = 0.5*dc['Io']
    sys = signal.TransferFunction(num, den)
    re = np.roots(den).real
    lhp = np.abs(re[re < -1e-3])
    tmax = 10.0/np.min(lhp) if lhp.size else 30.0/P['fsw']
    unstable = np.any(re > 1e-3)
    if unstable:
        tmax = min(tmax, 3.0/np.max(re[re > 1e-3]))
    tpos = np.linspace(0, tmax, N)
    _, vo = signal.step(sys, T=tpos)
    vo = -dI*vo
    tneg = np.linspace(-0.1*tmax, 0, max(8, N//10), endpoint=False)
    t = np.concatenate([tneg, tpos])
    v = np.concatenate([np.zeros_like(tneg), vo])
    ii = np.concatenate([np.full_like(tneg, dc['Io']), np.full_like(tpos, dc['Io']+dI)])
    return t, ii, v, dI, unstable

In [ ]:
def _sub(lbl):
    if '_' in lbl:
        base, sub = lbl.split('_', 1)
        return f'{base}<tspan baseline-shift="sub" font-size="0.7em">{sub}</tspan>'
    return lbl


def buck_schematic_svg(colors=None):
    """IEEE/ANSI-style synchronous buck schematic as a portable SVG string.
       colors={'Vin':..,'SW':..,'L':..,'C':..,'R':..} tints a component's
       symbol + label (wires stay black). No external deps."""
    colors = colors or {}
    S = '#171717'
    cV  = colors.get('Vin', S)
    cSW = colors.get('SW',  S)
    cL  = colors.get('L',   S)
    cC  = colors.get('C',   S)
    cR  = colors.get('R',   S)
    sw = 2
    def line(x1, y1, x2, y2, c=S, dash=None):
        d = f' stroke-dasharray="{dash}"' if dash else ''
        return (f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{c}" '
                f'stroke-width="{sw}" stroke-linecap="round"{d}/>')
    def dot(x, y, r=3.2, c=S):
        return f'<circle cx="{x}" cy="{y}" r="{r}" fill="{c}"/>'
    def gnd(x, y, c=S):
        return (line(x, y, x, y+8, c) +
                f'<line x1="{x-11}" y1="{y+8}" x2="{x+11}" y2="{y+8}" stroke="{c}" stroke-width="{sw}"/>'
                f'<line x1="{x-7}" y1="{y+13}" x2="{x+7}" y2="{y+13}" stroke="{c}" stroke-width="{sw}"/>'
                f'<line x1="{x-3}" y1="{y+18}" x2="{x+3}" y2="{y+18}" stroke="{c}" stroke-width="{sw}"/>')
    def txt(x, y, s, sz=15, style='italic', anchor='middle', c=S):
        return (f'<text x="{x}" y="{y}" font-family="Georgia, \'Times New Roman\', serif" '
                f'font-style="{style}" font-size="{sz}" fill="{c}" text-anchor="{anchor}">{s}</text>')
    def nmos(x, ytop, ybot, c):
        """Simplified NMOS: drain at (x,ytop), source at (x,ybot), gate lead left."""
        ym = 0.5*(ytop+ybot)
        p = [line(x, ytop, x, ym-20, c),                 # drain lead
             line(x, ym-20, x-8, ym-20, c),
             line(x-8, ym-17, x-8, ym+17, c),            # channel
             line(x-8, ym+20, x, ym+20, c),
             line(x, ym+20, x, ybot, c),                 # source lead
             line(x-8, ym+20, x-8, ym+17, c),
             line(x-8, ym-20, x-8, ym-17, c),
             line(x-14, ym-14, x-14, ym+14, c),          # gate plate
             line(x-26, ym, x-14, ym, c)]                # gate lead
        return ''.join(p), (x-26, ym)

    p = []
    # ---- Vin source (left) ----
    sx, sy, r = 62, 168, 24
    p.append(f'<circle cx="{sx}" cy="{sy}" r="{r}" fill="none" stroke="{cV}" stroke-width="{sw}"/>')
    p.append(txt(sx, sy-6, '+', 14, 'normal', 'middle', cV))
    p.append(txt(sx, sy+16, '\u2212', 14, 'normal', 'middle', cV))
    p.append(txt(sx-34, sy+5, _sub('V_in'), 16, 'italic', 'middle', cV))
    p.append(line(sx, sy-r, sx, 70)); p.append(line(sx, 70, 152, 70))
    p.append(line(sx, sy+r, sx, 272)); p.append(gnd(sx, 272))

    # ---- high-side / low-side FETs (vertical stack at x=152) ----
    hs, hg = nmos(152, 70, 138, cSW); p.append(hs)
    p.append(txt(168, 100, _sub('Q_H'), 13, 'italic', 'start', cSW))
    p.append(dot(152, 138)); p.append(txt(160, 152, 'SW', 11, 'normal', 'start'))
    ls, lg = nmos(152, 138, 218, cSW); p.append(ls)
    p.append(txt(168, 180, _sub('Q_L'), 13, 'italic', 'start', cSW))
    p.append(gnd(152, 218))

    # ---- inductor L (rL) ----
    p.append(line(152, 138, 214, 138))
    arcs = ''.join(f'A11,11 0 0 1 {214+22*(i+1)},138 ' for i in range(4))
    p.append(f'<path d="M214,138 {arcs}" fill="none" stroke="{cL}" stroke-width="{sw}"/>')
    p.append(txt(258, 118, _sub('L') + '&#8201;(' + _sub('r_L') + ')', 14, 'italic', 'middle', cL))
    p.append(line(302, 138, 364, 138)); p.append(dot(364, 138))

    # ---- output capacitor with ESR ----
    p.append(line(364, 138, 364, 168))
    p.append(line(352, 168, 376, 168, cC)); p.append(line(352, 178, 376, 178, cC))
    p.append(txt(384, 178, _sub('C'), 14, 'italic', 'start', cC))
    p.append(line(364, 178, 364, 192))
    zig = '364,192 370,196 358,204 370,212 358,220 364,224'
    p.append(f'<polyline points="{zig}" fill="none" stroke="{cC}" stroke-width="{sw}" stroke-linejoin="round"/>')
    p.append(txt(380, 212, _sub('r_C'), 13, 'italic', 'start', cC))
    p.append(line(364, 224, 364, 240)); p.append(gnd(364, 240))

    # ---- load ----
    p.append(line(364, 138, 452, 138)); p.append(dot(452, 138))
    p.append(line(452, 138, 452, 162))
    zig2 = '452,162 458,166 446,174 458,182 446,190 458,198 446,206 452,210'
    p.append(f'<polyline points="{zig2}" fill="none" stroke="{cR}" stroke-width="{sw}" stroke-linejoin="round"/>')
    p.append(txt(468, 190, _sub('R_load'), 14, 'italic', 'start', cR))
    p.append(line(452, 210, 452, 240)); p.append(gnd(452, 240))
    p.append(line(452, 138, 520, 138))
    p.append(txt(530, 143, _sub('V_out'), 16, 'italic', 'start', cR))

    # ---- PWM + EA block with feedback (dashed) ----
    p.append(f'<rect x="216" y="252" width="120" height="40" fill="none" stroke="{cSW}" stroke-width="{sw}" rx="4"/>')
    p.append(txt(276, 277, 'PWM + EA', 12, 'normal', 'middle', cSW))
    p.append(line(452, 138, 452, 272, S, '5,4')); p.append(line(452, 272, 336, 272, S, '5,4'))
    p.append(line(216, 272, 104, 272, S, '5,4'))
    p.append(line(104, 272, 104, hg[1], S, '5,4')); p.append(line(104, hg[1], hg[0], hg[1], S, '5,4'))
    p.append(line(104, lg[1], lg[0], lg[1], S, '5,4'))

    body = '\n  '.join(p)
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 305" '
            f'width="600" height="305" font-family="serif">\n  {body}\n</svg>')


ASCII_SCHEMATIC = r"""
  Vin o--[QH]--+--(L, rL)--+------+---o Vout
               |           |      |
             [QL]         C      Rload
               |          rC      |
              GND          |      |
                          GND    GND
        PWM+EA drives QH/QL from Vout feedback
"""


def schematic_widget():
    """Return the schematic as an ipywidget (SVG by default, ASCII only if SVG fails)."""
    import ipywidgets as w
    try:
        svg = buck_schematic_svg()
        if not svg.lstrip().startswith('<svg'):
            raise ValueError('SVG generation produced no <svg> root')
        return w.HTML(value=svg)
    except Exception as e:
        return w.HTML(value=f'<pre>[SVG failed: {e}]\n{ASCII_SCHEMATIC}</pre>')

In [ ]:
class BuckExplorer:
    GHOST_ALPHA = [1.0, 0.42, 0.18]
    GHOST_LS    = ['-', '--', ':']
    GHOST_LW    = [1.8, 1.2, 1.0]

    # each freq panel draws one or more series: (response-key, color, legend-label)
    FREQ = {
        'olm': dict(ylabel=r'$|G_p|,\ |1/G_c|$ (dB)', step=20, phase=False,
                    title=r'Open-loop: plant $G_p$ vs comp $1/G_c$',
                    series=[('Gp_mag', 'k', r'$G_p$'), ('invGc_mag', '0.55', r'$1/G_c$')]),
        'clm': dict(ylabel=r'$|T|$ (dB)', step=20, phase=False,
                    title=r'Loop gain  $T = G_p\,G_c$',
                    series=[('T_mag', 'k', r'$|T|$')]),
        'olp': dict(ylabel=r'$\angle G_p,\ \angle(1/G_c){-}180\degree$ (°)', step=90, phase=True,
                    title=None,
                    series=[('Gp_ph', 'k', r'$\angle G_p$'),
                            ('invGc_ph_s', '0.55', r'$\angle(1/G_c){-}180\degree$')]),
        'clp': dict(ylabel=r'$\angle T$ (°)', step=90, phase=True,
                    title=None,
                    series=[('T_ph', 'k', r'$\angle T$')]),
    }
    TIME = ('ist', 'vst', 'ilw', 'vrw')
    TGRP = {'ist': 'step', 'vst': 'step', 'ilw': 'sw', 'vrw': 'sw'}

    # component -> highlight colour (rainbow). Tokens tagged with a component key
    # get this colour when that component is active; everything else stays black.
    COMP_COLORS = {'Vin': '#e6194B', 'SW': '#3a6ee0', 'L': '#f58231',
                   'C': '#2a9d3a', 'R': '#911eb4'}

    # Formula ledger node trees (('t',mathtext,compkey) | ('frac',[num],[den]) | ('sqrt',[inner]))
    LEDGER = [
        ('Vin',  [('t', r'V_{in}', 'Vin')],                lambda v: _fnum(v, '{:.3g}', ' V')),
        ('Vout', [('t', r'V_{out}', 'R')],                 lambda v: _fnum(v, '{:.3g}', ' V')),
        ('L',    [('t', r'L', 'L')],                       lambda v: _eng(v, 'H')),
        ('C',    [('t', r'C', 'C')],                       lambda v: _eng(v, 'F')),
        ('rC',   [('t', r'r_C\,(\mathrm{ESR})', 'C')],     lambda v: _eng(v, '\u03a9')),
        ('rL',   [('t', r'r_L\,(\mathrm{DCR})', 'L')],     lambda v: _eng(v, '\u03a9')),
        ('fsw',  [('t', r'f_{sw}', 'SW')],                 lambda v: _eng(v, 'Hz')),
        ('R',    [('t', r'R_{load}', 'R')],                lambda v: _fnum(v, '{:.3g}', ' \u03a9')),
        ('Ri',   [('t', r'R_i\,(\mathrm{sense})', 'SW')],  lambda v: _fnum(v, '{:.3g}', ' V/A')),
        ('Se',   [('t', r'S_e\,(\mathrm{slope\ comp.})', 'SW')],
                                                           lambda v: _eng(v, 'V/s')),
        ('fct',  [('t', r'f_{c,t}\,(\mathrm{target})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('M',    [('t', r'M\!=\!', ''),
                  ('frac', [('t', r'V_{out}', 'R')], [('t', r'V_{in}', 'Vin')])],
                                                           lambda v: _fnum(v, '{:.3f}')),
        ('K',    [('t', r'K\!=\!', ''),
                  ('frac', [('t', r'2\,', ''), ('t', r'L', 'L'), ('t', r'\,f_{sw}', 'SW')],
                           [('t', r'R_{load}', 'R')])],    lambda v: _fnum(v, '{:.3f}')),
        ('Kcrit', [('t', r'K_{crit}\!=\!1\!-\!M', '')],    lambda v: _fnum(v, '{:.3f}')),
        ('mode', [('t', r'\mathrm{mode}\ (K \gtrless K_{crit})', '')],
                                                           lambda v: 'CCM' if v < 1.5 else 'DCM'),
        ('D',    [('t', r'D\ (\mathrm{duty})', 'SW')],     lambda v: _fnum(v, '{:.3f}')),
        ('dIL',  [('t', r'\Delta I_L\!=\!', ''),
                  ('frac', [('t', r'V_{out}', 'R'), ('t', r'(1{-}D)', 'SW')],
                           [('t', r'L', 'L'), ('t', r'\,f_{sw}', 'SW')])],
                                                           lambda v: _eng(v, 'A')),
        ('dVo',  [('t', r'\Delta V_{out}\!\approx\!\Delta I_L\,(', ''),
                  ('t', r'r_C', 'C'), ('t', r'{+}', ''),
                  ('frac', [('t', r'1', '')],
                           [('t', r'8\,', ''), ('t', r'C', 'C'), ('t', r'f_{sw}', 'SW')]),
                  ('t', r')', '')],                        lambda v: _eng(v, 'V')),
        ('f0',   [('t', r'f_0\!=\!', ''),
                  ('frac', [('t', r'1', '')],
                           [('t', r'2\pi', ''),
                            ('sqrt', [('t', r'L', 'L'), ('t', r'\,C', 'C')])])],
                                                           lambda v: _eng(v, 'Hz')),
        ('Q',    [('t', r'Q\ (\mathrm{LC})', '')],         lambda v: _fnum(v, '{:.2f}')),
        ('fesr', [('t', r'f_{ESR}\!=\!', ''),
                  ('frac', [('t', r'1', '')],
                           [('t', r'2\pi\,', ''), ('t', r'r_C', 'C'), ('t', r'\,C', 'C')])],
                                                           lambda v: _eng(v, 'Hz')),
        ('fp',   [('t', r'f_p\ (\mathrm{plant\ dom.\ pole})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('fz1',  [('t', r'f_{z1},\,f_{z2}\ (\mathrm{comp\ zeros})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('fz2',  [('t', r'\;\;\;f_{z2}\ (\mathrm{comp})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('fpc2', [('t', r'f_{p2},\,f_{p3}\ (\mathrm{comp\ poles})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('fpc3', [('t', r'\;\;\;f_{p3}\ (\mathrm{comp})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('Qp',   [('t', r'Q_p\!=\!', ''),
                  ('frac', [('t', r'1', '')],
                           [('t', r'\pi\,(', ''), ('t', r'm_c', 'SW'), ('t', r'(1{-}D){-}0.5)', 'SW')])],
                                                           lambda v: _fnum(v, '{:.2f}')),
        ('fc',   [('t', r'f_c\ (\mathrm{loop\ crossover})', '')],
                                                           lambda v: _eng(v, 'Hz')),
        ('pm',   [('t', r'\mathrm{PM}', '')],              lambda v: _fnum(v, '{:.1f}', '\u00b0')),
        ('gm',   [('t', r'\mathrm{GM}', '')],              lambda v: _fnum(v, '{:.1f}', ' dB')),
    ]

    def __init__(self):
        plt.ioff()
        # ---- main figure: 8 plot panels ----
        self.fig = plt.figure(figsize=(13, 13))
        try:
            self.fig.canvas.header_visible = False
            self.fig.canvas.toolbar_visible = True
        except Exception:
            pass
        gs = self.fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1],
                                   hspace=0.46, wspace=0.26,
                                   left=0.08, right=0.95, top=0.96, bottom=0.05)
        self.ax = {
            'ist': self.fig.add_subplot(gs[0, 0]),
            'vst': self.fig.add_subplot(gs[0, 1]),
            'ilw': self.fig.add_subplot(gs[1, 0]),
            'vrw': self.fig.add_subplot(gs[1, 1]),
            'olm': self.fig.add_subplot(gs[2, 0]),
            'clm': self.fig.add_subplot(gs[2, 1]),
            'olp': self.fig.add_subplot(gs[3, 0]),
            'clp': self.fig.add_subplot(gs[3, 1]),
        }
        # ---- equations: rendered to a static, auto-updating image ----
        from matplotlib.figure import Figure as _Figure
        from matplotlib.backends.backend_agg import FigureCanvasAgg as _AggCanvas
        import io as _io, ipywidgets as _w
        self._io = _io
        self.fig_eq = _Figure(figsize=(6.8, 11.4), dpi=100); _AggCanvas(self.fig_eq)
        self.ax_eq = self.fig_eq.add_axes([0, 0, 1, 1]); self.ax_eq.axis('off')
        self.eq_img = _w.Image(format='png')
        try:
            self.sch_widget = _w.HTML(value=buck_schematic_svg())
        except Exception as e:
            self.sch_widget = _w.HTML(value=f'<pre>[SVG failed: {e}]\n{ASCII_SCHEMATIC}</pre>')
        self.warn_widget = _w.HTML(value='')

        self.history_tr = deque(maxlen=3)          # load-step (t, ii, vo, dI)
        self.history_sw = deque(maxlen=3)          # switching waves (t, iL, vac)
        self.history_bd = deque(maxlen=3)          # bode dicts
        self.history_an = deque(maxlen=3)          # crossover annotations
        self.lims    = {}
        self._curves = {}
        self.eq_hist = {k: deque(maxlen=2) for k, *_ in self.LEDGER}
        self._prev_si   = None
        self._changed   = None
        self._bold_keys = set()
        self._cursors = {}
        self._sticky  = {k: [] for k in (*self.TIME, *self.FREQ)}
        self._fc = self._pm = self._gm = None
        self._f0 = self._fesr = self._fsw = None
        self._mode = self._ckind = None
        self._fp = self._dI = None
        self.color_state = None
        self._connect_events()

    # ---------------------------------------------------------------- limits
    def _expand(self, key, x, y, logx=False):
        x = np.asarray(x, float); y = np.asarray(y, float)
        m = np.isfinite(y)
        if logx:
            mx = np.isfinite(x) & (x > 0); x = x[mx]; y = y[mx]; m = np.isfinite(y)
        if not np.any(m): return
        xn, xx = np.nanmin(x), np.nanmax(x)
        yn, yx = np.nanmin(y[m]), np.nanmax(y[m])
        if key in self.lims:
            l = self.lims[key]
            # x: keep the same range (set once); y: grow only
            l[2] = min(l[2], yn); l[3] = max(l[3], yx)
        else:
            self.lims[key] = [xn, xx, yn, yx]

    def _apply_limits(self, key, ax, logx=False, snap=None):
        if key not in self.lims: return
        xn, xx, yn, yx = self.lims[key]
        if logx:
            if xn > 0: ax.set_xlim(xn, xx)
        else:
            pad = 0.04*(xx-xn or 1); ax.set_xlim(xn-pad, xx+pad)
        if snap:
            ax.set_ylim(np.floor(yn/snap)*snap, np.ceil(yx/snap)*snap)
        else:
            pad = 0.10*(yx-yn or 1); ax.set_ylim(yn-pad, yx+pad)

    # ---------------------------------------------------------------- diff shade
    def _shade(self, ax, xn_new, y_new, xn_old, y_old, logx=False):
        """Green where new>=old, red where new<old, on a common x grid."""
        if logx:
            lo = max(np.min(xn_new), np.min(xn_old))
            hi = min(np.max(xn_new), np.max(xn_old))
            if hi <= lo: return
            xg = np.logspace(np.log10(lo), np.log10(hi), 600)
            v_new = np.interp(np.log10(xg), np.log10(xn_new), y_new)
            v_old = np.interp(np.log10(xg), np.log10(xn_old), y_old)
        else:
            xg = np.union1d(xn_new, xn_old)
            v_new = np.interp(xg, xn_new, y_new)
            v_old = np.interp(xg, xn_old, y_old)
        ax.fill_between(xg, v_old, v_new, where=(v_new >= v_old), interpolate=True,
                        color='#27ae60', alpha=0.20, zorder=0)
        ax.fill_between(xg, v_old, v_new, where=(v_new < v_old), interpolate=True,
                        color='#c0392b', alpha=0.20, zorder=0)

    # ---------------------------------------------------------------- transients
    def _draw_time(self, pid, ylabel, title):
        ax = self.ax[pid]; ax.cla()
        cur = self._curves.get(pid, [])
        if len(cur) >= 2:
            (xo, yo), (xn, yn) = cur[-2], cur[-1]
            self._shade(ax, xn, yn, xo, yo, logx=False)
        n = len(cur)
        for i, (x, y) in enumerate(cur):
            age = n-1-i
            ax.plot(x, y, color='k', alpha=self.GHOST_ALPHA[age],
                    ls=self.GHOST_LS[age], lw=self.GHOST_LW[age], zorder=6-age)
        if self.TGRP[pid] == 'step':
            ax.axvline(0.0, color='0.7', lw=0.7, ls=':', zorder=1)
        ax.set_xlabel(r'$t$ (µs)'); ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=10); ax.grid(alpha=0.3, which='both')
        self._apply_limits(pid, ax)

    # ---------------------------------------------------------------- bode
    def _draw_freq(self, pid):
        spec = self.FREQ[pid]; ax = self.ax[pid]; ax.cla()
        H = self.history_bd; n = len(H)
        if n >= 2:
            for subkey, color, lab in spec['series']:
                self._shade(ax, H[-1]['f'], H[-1][subkey],
                            H[-2]['f'], H[-2][subkey], logx=True)
        for i, Rr in enumerate(H):
            age = n-1-i
            for subkey, color, lab in spec['series']:
                ax.semilogx(Rr['f'], Rr[subkey], color=color,
                            alpha=self.GHOST_ALPHA[age], ls=self.GHOST_LS[age],
                            lw=self.GHOST_LW[age], zorder=6-age,
                            label=(lab if age == n-1 else None))
        if self._fc and np.isfinite(self._fc):
            ax.axvline(self._fc, color='0.55', lw=0.8, ls='--', zorder=1)
        if self._fsw:
            ax.axvline(self._fsw/2, color='0.75', lw=0.8, ls=':', zorder=1)
        f0m = self._f0 if self._mode == 'CCM' else self._fp
        if pid in ('olm', 'olp') and f0m and np.isfinite(f0m):
            ax.axvline(f0m, color='0.75', lw=0.8, ls=':', zorder=1)
        ax.axhline(-180 if spec['phase'] else 0, color='0.6', lw=0.7,
                   ls=':' if spec['phase'] else '--', zorder=1)
        ax.yaxis.set_major_locator(MultipleLocator(spec['step']))
        ax.set_ylabel(spec['ylabel']); ax.set_xlabel(r'$f$ (Hz)')
        ax.grid(alpha=0.3, which='both')
        if len(spec['series']) > 1:
            ax.legend(loc='best', fontsize=8, framealpha=0.9)
        if spec['title']:
            ttl = spec['title']
            if pid == 'olm' and self._pm is not None:
                ff = (f"$f_0$={_eng(self._f0,'Hz')}" if self._mode == 'CCM'
                      else f"$f_p$={_eng(self._fp,'Hz')}")
                ttl += (f"   ({self._mode}, {self._ckind})\n{ff} · "
                        f"$f_{{ESR}}$={_eng(self._fesr,'Hz')} · "
                        f"$f_{{sw}}/2$={_eng(self._fsw/2,'Hz')}")
            elif pid == 'clm' and self._pm is not None:
                ttl += (f"\n$f_c$={_eng(self._fc,'Hz')} · PM={_fnum(self._pm,'{:.0f}','°')}"
                        f" · GM={_fnum(self._gm,'{:.1f}',' dB')}")
            ax.set_title(ttl, fontsize=9)
        # ---- phase-margin gap arrow (olp): fades/dashes like the ghost curves ----
        if pid == 'olp' and self.history_an:
            Han = self.history_an; nan = len(Han)
            for i, an in enumerate(Han):
                age = nan-1-i
                al = self.GHOST_ALPHA[age]; ls = self.GHOST_LS[age]
                fc = an['fc']
                if not fc or not np.isfinite(fc):
                    continue
                yA, yB = an['yA_ph'], an['yB_ph']
                ax.annotate('', xy=(fc, yA), xytext=(fc, yB),
                            arrowprops=dict(arrowstyle='<->', color='#c0392b', lw=1.6,
                                            ls=ls, alpha=al), zorder=9-age)
                if age == 0:
                    ax.annotate(f"PM={an['pm']:.0f}°", xy=(fc, 0.5*(yA+yB)),
                                xytext=(7, 0), textcoords='offset points', fontsize=8.5,
                                color='#c0392b', fontweight='bold', va='center', ha='left',
                                zorder=10, bbox=dict(boxstyle='round,pad=0.2', fc='white',
                                                     ec='#c0392b', alpha=0.92))
        self._apply_limits(pid, ax, logx=True,
                           snap=(spec['step'] if spec['phase'] else None))

    # ---------------------------------------------------------------- ledger
    def _active_colors(self):
        st = self.color_state
        if st is None:
            return {}
        if st == 'ALL':
            return dict(self.COMP_COLORS)
        return {st: self.COMP_COLORS[st]}

    def set_color(self, state):
        """Set highlight state (None=off, 'ALL', or a component key) and repaint
        the SVG schematic and the formula ledger."""
        if state == self.color_state:
            return
        self.color_state = state
        cm = self._active_colors()
        try:
            self.sch_widget.value = buck_schematic_svg(colors=cm)
        except Exception:
            pass
        self._draw_equations()

    def _expr_size(self, ax, rend, AW, AH, expr, fs):
        """Return (width, ascent, descent) of a node list, in axes fraction."""
        w = 0.0; asc = desc = 0.0
        for node in expr:
            if node[0] == 't':
                t = ax.text(0, 0, '$' + node[1] + '$', fontsize=fs)
                bb = t.get_window_extent(rend); t.remove()
                nw, nh = bb.width/AW, bb.height/AH
                na = nd = nh/2.0
            elif node[0] == 'frac':
                wn, an, dn = self._expr_size(ax, rend, AW, AH, node[1], fs*0.95)
                wd, ad, dd = self._expr_size(ax, rend, AW, AH, node[2], fs*0.95)
                nw = max(wn, wd) + 0.012
                na = 0.005 + (an + dn); nd = 0.005 + (ad + dd)
            else:  # sqrt
                wi, ai, di = self._expr_size(ax, rend, AW, AH, node[1], fs)
                radw = 0.5*(ai + di)
                nw = radw + wi + 0.004; na = ai + 0.007; nd = di
            w += nw; asc = max(asc, na); desc = max(desc, nd)
        return w, asc, desc

    def _expr_draw(self, ax, rend, AW, AH, expr, x, yc, fs, colors):
        """Draw a node list starting at left x, vertically centred on yc."""
        for node in expr:
            if node[0] == 't':
                col = colors.get(node[2], 'k')
                t = ax.text(x, yc, '$' + node[1] + '$', fontsize=fs, va='center',
                            ha='left', color=col)
                x += t.get_window_extent(rend).width/AW
            elif node[0] == 'frac':
                wn, an, dn = self._expr_size(ax, rend, AW, AH, node[1], fs*0.95)
                wd, ad, dd = self._expr_size(ax, rend, AW, AH, node[2], fs*0.95)
                w = max(wn, wd) + 0.012
                ax.plot([x, x+w], [yc, yc], color='k', lw=1.1, solid_capstyle='butt', zorder=3)
                self._expr_draw(ax, rend, AW, AH, node[1],
                                x + (w-wn)/2, yc + 0.005 + dn, fs*0.95, colors)
                self._expr_draw(ax, rend, AW, AH, node[2],
                                x + (w-wd)/2, yc - 0.005 - ad, fs*0.95, colors)
                x += w
            else:  # sqrt
                wi, ai, di = self._expr_size(ax, rend, AW, AH, node[1], fs)
                radw = 0.5*(ai + di); top = yc + ai + 0.006
                ax.plot([x, x+0.42*radw, x+radw, x+radw+wi],
                        [yc, yc-di, top, top], color='k', lw=1.1,
                        solid_joinstyle='miter', zorder=3)
                self._expr_draw(ax, rend, AW, AH, node[1], x+radw, yc, fs, colors)
                x += radw + wi + 0.004
        return x

    def _draw_equations(self):
        ax = self.ax_eq; fig = self.fig_eq
        ax.cla(); ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        fig.canvas.draw()
        rend = fig.canvas.get_renderer()
        AW = ax.get_window_extent(rend).width
        AH = ax.get_window_extent(rend).height
        def wfrac(t): return t.get_window_extent(rend).width  / AW
        def hfrac(t): return t.get_window_extent(rend).height / AH
        colors = self._active_colors()
        FS = 10.5

        ax.text(0.5, 0.995, 'Formula  $=$ value  $=$ previous',
                ha='center', va='top', fontsize=8.5, color='0.45')
        x0, gap = 0.03, 0.024

        # PASS 1 - measure so both "=" columns align vertically
        rows = []; maxwf = maxwv = 0.0
        for k, expr, fmt in self.LEDGER:
            hist = self.eq_hist[k]
            if not hist:
                continue
            bold = (k == self._changed) or (k in self._bold_keys)
            wf, asc, desc = self._expr_size(ax, rend, AW, AH, expr, FS)
            t2 = ax.text(0, 0, '$=$ ' + fmt(hist[-1]), fontsize=FS,
                         fontweight=('bold' if bold else 'normal'))
            wv = wfrac(t2); t2.remove()
            rows.append(dict(k=k, expr=expr, fmt=fmt, hist=hist, bold=bold,
                             wf=wf, asc=asc, desc=desc))
            maxwf = max(maxwf, wf); maxwv = max(maxwv, wv)
        vx = x0 + maxwf + gap          # first "=" column (value)
        ox = vx + maxwv + gap          # second "=" column (previous)

        # PASS 2 - place everything
        y = 0.975
        for r in rows:
            k, fmt, hist = r['k'], r['fmt'], r['hist']
            yc = y - r['asc']
            self._expr_draw(ax, rend, AW, AH, r['expr'], x0, yc, FS, colors)
            tv = ax.text(vx, yc, '$=$ ' + fmt(hist[-1]), fontsize=FS, va='center',
                         ha='left', color='k', fontweight=('bold' if r['bold'] else 'normal'))
            wv, hv = wfrac(tv), hfrac(tv)
            if k == self._changed:
                tv.set_bbox(dict(boxstyle='round,pad=0.18', fc='#ffe680', ec='none', alpha=0.9))
            yu = yc - hv/2.0 - 0.003
            ax.plot([vx, vx+wv], [yu, yu], color='k', lw=1.3, solid_capstyle='butt', zorder=3)
            if len(hist) >= 2:
                to = ax.text(ox, yc, '$=$ ' + fmt(hist[-2]), fontsize=FS-0.5,
                             va='center', ha='left', color='k')
                wo, ho = wfrac(to), hfrac(to)
                yuo = yc - ho/2.0 - 0.003
                ax.plot([ox, ox+wo], [yuo, yuo], color='k', lw=1.0, ls=(0, (4, 3)), zorder=3)
            y -= (r['asc'] + r['desc'] + 0.013)
        buf = self._io.BytesIO()
        from matplotlib.transforms import Bbox
        fig.canvas.draw()
        rend = fig.canvas.get_renderer()
        arts = list(ax.texts) + list(ax.lines)
        exts = [a.get_window_extent(rend) for a in arts if a.get_visible()]
        exts = [e for e in exts if e.width > 0 and e.height > 0]
        if exts:
            bb = Bbox.union(exts)
            pad = 8
            bb = Bbox.from_extents(bb.x0-pad, bb.y0-pad, bb.x1+pad, bb.y1+pad)
            bb_in = bb.transformed(fig.dpi_scale_trans.inverted())
            fig.savefig(buf, format='png', dpi=100, bbox_inches=bb_in, facecolor='white')
        else:
            fig.savefig(buf, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        self.eq_img.value = buf.getvalue()

    # ---------------------------------------------------------------- update
    def update(self, P):
        dc = solve_dc(P)
        lp = loop_polys(P, dc)
        an = analyze(lp, P['fsw'])
        R = responses(P, dc, lp)
        ts, ii, vs, dI, unstable = load_step(P, dc, lp)
        tw, iL, vac = ripple_waves(P, dc)
        self._fc, self._pm, self._gm = an['fc'], an['pm'], an['gm']
        self._fsw = P['fsw']
        self._mode = dc['mode']; self._ckind = lp['comp']['kind']
        self._fesr = lp['pinfo']['fesr']
        self._fp = lp['pinfo'].get('fp', np.nan)
        self._dI = dI
        # LC resonance/Q of the power stage (always physical, control-independent)
        a1 = (P['L'] + P['C']*(P['rC']*P['Rload'] + P['rC']*P['rL'] + P['rL']*P['Rload']))/(P['Rload']+P['rL'])
        a2 = P['L']*P['C']*(P['Rload']+P['rC'])/(P['Rload']+P['rL'])
        self._f0 = 1.0/(2*np.pi*np.sqrt(a2))
        Qlc = np.sqrt(a2)/a1

        # phase-gap annotation at crossover (PM arrow on olp)
        if np.isfinite(self._fc):
            _lf = np.log10(R['f']); _lc = np.log10(self._fc)
            self.history_an.append(dict(
                fc=self._fc, pm=self._pm,
                yA_ph=float(np.interp(_lc, _lf, R['Gp_ph'])),
                yB_ph=float(np.interp(_lc, _lf, R['invGc_ph_s']))))
        self.history_tr.append((ts, ii, vs))
        self.history_sw.append((tw, iL, vac))
        self.history_bd.append(R)

        # rebuild time-panel curve lists (oldest..newest)
        self._curves = {pid: [] for pid in self.TIME}
        for (tt, jj, vv) in self.history_tr:
            self._curves['ist'].append((tt*1e6, jj))
            self._curves['vst'].append((tt*1e6, vv*1e3))
        for (tt, jj, vv) in self.history_sw:
            self._curves['ilw'].append((tt*1e6, jj))
            self._curves['vrw'].append((tt*1e6, vv*1e3))

        # grow limits
        self._expand('ist', ts*1e6, ii); self._expand('vst', ts*1e6, vs*1e3)
        self._expand('ilw', tw*1e6, iL); self._expand('vrw', tw*1e6, vac*1e3)
        for pid, spec in self.FREQ.items():
            for subkey, color, lab in spec['series']:
                self._expand(pid, R['f'], R[subkey], logx=True)

        # ---- which input changed, and by what factor x ----
        si = {k: P[k] for k in ('Vin', 'Vout', 'L', 'C', 'rC', 'rL',
                                'fsw', 'Rload', 'Ri', 'Se', 'fct')}
        self._changed = None; self._bold_keys = set(); lnx = 0.0
        if self._prev_si is not None:
            chg = [(k, si[k]/self._prev_si[k]) for k in si
                   if self._prev_si[k] > 0 and si[k] > 0
                   and abs(np.log(si[k]/self._prev_si[k])) > 1e-9]
            if chg:
                k, f = max(chg, key=lambda kf: abs(np.log(kf[1])))
                self._changed = {'Rload': 'R'}.get(k, k)
                lnx = abs(np.log(f))

        zs, ps = lp['comp']['zs'], lp['comp']['ps']
        vals = dict(Vin=P['Vin'], Vout=dc['Vout'], L=P['L'], C=P['C'], rC=P['rC'],
                    rL=P['rL'], fsw=P['fsw'], R=P['Rload'], Ri=P['Ri'], Se=P['Se'],
                    fct=P['fct'], M=dc['M'], K=dc['K'], Kcrit=dc['Kcrit'],
                    mode=(1.0 if dc['mode'] == 'CCM' else 2.0), D=dc['D'],
                    dIL=dc['dIL'], dVo=dc['dVout'], f0=self._f0, Q=Qlc,
                    fesr=self._fesr, fp=self._fp,
                    fz1=zs[0], fz2=(zs[1] if len(zs) > 1 else np.nan),
                    fpc2=ps[0], fpc3=(ps[1] if len(ps) > 1 else np.nan),
                    Qp=lp['pinfo'].get('Qp', np.nan),
                    fc=self._fc, pm=self._pm, gm=self._gm)
        if lnx > 1e-9:
            for k in vals:
                h = self.eq_hist[k]
                if h and h[-1] and vals[k] and np.isfinite(h[-1]) and np.isfinite(vals[k]) \
                   and h[-1] > 0 and vals[k] > 0:
                    if abs(np.log(vals[k]/h[-1])) > 0.6*lnx:
                        self._bold_keys.add(k)
        for k in self.eq_hist: self.eq_hist[k].append(vals[k])
        self._prev_si = si

        # ---- warnings banner ----
        warns = list(dc['warn']) + list(lp['warn'])
        if unstable:
            warns.append('closed loop UNSTABLE — load-step response diverges')
        if warns:
            body = '<br>'.join('&#9888;&#65039; ' + w for w in warns)
            self.warn_widget.value = (f'<div style="color:#b71c1c;font-weight:bold;'
                                      f'font-size:13px;padding:4px 0">{body}</div>')
        else:
            self.warn_widget.value = ('<div style="color:#2e7d32;font-size:13px;'
                                      'padding:4px 0">&#9989; operating point OK '
                                      f'({dc["mode"]}, {lp["comp"]["kind"]}, '
                                      f'PM {_fnum(self._pm, "{:.0f}", "&deg;")})</div>')

        self._clear_cursors()
        for k in self._sticky: self._sticky[k] = []
        self._draw_time('ist', r'$I_{load}$ (A)',
                        f'Load — current step (+{dI:.2g} A = 50%)')
        self._draw_time('vst', r'$\Delta V_{out}$ (mV)',
                        'Output — deviation (closed loop)')
        self._draw_time('ilw', r'$i_L$ (A)',
                        f'Inductor current — 3 cycles ({dc["mode"]}, '
                        f'$\\Delta I_L$={_eng(dc["dIL"],"A")})')
        self._draw_time('vrw', r'$v_{out}$ ripple (mV)',
                        f'Output ripple ($\\Delta V_{{out}}\\approx${_eng(dc["dVout"],"V")})')
        for pid in self.FREQ: self._draw_freq(pid)
        self._draw_equations()
        self.fig.canvas.draw_idle()

    def reset(self):
        self.history_tr.clear(); self.history_sw.clear()
        self.history_bd.clear(); self.history_an.clear()
        self.lims.clear(); self._curves = {}
        for k in self.eq_hist: self.eq_hist[k].clear()
        self._prev_si = None; self._changed = None; self._bold_keys = set()
        self._clear_cursors()
        for k in self._sticky: self._sticky[k] = []

    # ============================================================ interactivity
    def _connect_events(self):
        c = self.fig.canvas
        c.mpl_connect('motion_notify_event', self._on_move)
        c.mpl_connect('button_press_event', self._on_click)
        self._ax2id = {self.ax[k]: k for k in (*self.TIME, *self.FREQ)}

    def _group(self, pid):
        return self.TGRP[pid] if pid in self.TGRP else 'freq'

    def _targets(self, pid):
        g = self._group(pid)
        if g == 'freq':
            return list(self.FREQ.keys())
        return [p for p in self.TIME if self.TGRP[p] == g]

    def _interp(self, xy, x, logx):
        xs, ys = xy
        if logx:
            return float(np.interp(np.log10(x), np.log10(xs), ys))
        return float(np.interp(x, xs, ys))

    def _fmt(self, pid, x, y, prefix=''):
        if pid == 'ist':  return f'{x:.3g} µs\n{y:.3g} A'
        if pid == 'vst':  return f'{x:.3g} µs\n{y:.3g} mV'
        if pid == 'ilw':  return f'{x:.3g} µs\n{y:.3g} A'
        if pid == 'vrw':  return f'{x:.3g} µs\n{y:.3g} mV'
        unit = '°' if self.FREQ[pid]['phase'] else 'dB'
        return f'{prefix}{_eng(x, "Hz")}\n{y:.1f} {unit}'

    def _clear_cursors(self):
        for arts in self._cursors.values():
            for a in arts:
                try: a.remove()
                except Exception: pass
        self._cursors = {}

    def _on_move(self, ev):
        if ev.inaxes not in self._ax2id or ev.xdata is None:
            if self._cursors:
                self._clear_cursors(); self.fig.canvas.draw_idle()
            return
        src = self._ax2id[ev.inaxes]
        self._clear_cursors()
        x = ev.xdata
        for pid in self._targets(src):
            ax = self.ax[pid]
            arts = [ax.axvline(x, color='0.25', lw=0.9, alpha=0.85, zorder=15)]
            if pid in self.TIME:
                cur = self._curves.get(pid, [])
                if not cur:
                    continue
                yi = self._interp(cur[-1], x, False)
                arts.append(ax.annotate(self._fmt(pid, x, yi), xy=(x, yi),
                            xytext=(8, 8), textcoords='offset points', fontsize=8,
                            bbox=dict(boxstyle='round,pad=0.3', fc='#fffbe6', ec='0.6',
                                      alpha=0.95), zorder=16))
                if len(cur) >= 2:                       # previous (dashed) curve label
                    yp = self._interp(cur[-2], x, False)
                    arts.append(ax.annotate(self._fmt(pid, x, yp), xy=(x, yp),
                                xytext=(8, -22), textcoords='offset points', fontsize=7.5,
                                color='0.45', bbox=dict(boxstyle='round,pad=0.25',
                                fc='#f2f2f2', ec='0.7', alpha=0.85), zorder=15))
            else:                                       # freq panel: one label per series
                H = self.history_bd
                if not H:
                    continue
                spec = self.FREQ[pid]; off = 8
                for subkey, color, lab in spec['series']:
                    yi = self._interp((H[-1]['f'], H[-1][subkey]), x, True)
                    pref = (lab + '  ') if len(spec['series']) > 1 else ''
                    arts.append(ax.annotate(self._fmt(pid, x, yi, pref), xy=(x, yi),
                                xytext=(8, off), textcoords='offset points', fontsize=8,
                                color=('0.35' if color != 'k' else 'k'),
                                bbox=dict(boxstyle='round,pad=0.3', fc='#fffbe6',
                                          ec='0.6', alpha=0.95), zorder=16))
                    off -= 30
                if len(spec['series']) == 1 and len(H) >= 2:   # previous curve label
                    sk = spec['series'][0][0]
                    yp = self._interp((H[-2]['f'], H[-2][sk]), x, True)
                    arts.append(ax.annotate(self._fmt(pid, x, yp), xy=(x, yp),
                                xytext=(8, -22), textcoords='offset points', fontsize=7.5,
                                color='0.45', bbox=dict(boxstyle='round,pad=0.25',
                                fc='#f2f2f2', ec='0.7', alpha=0.85), zorder=15))
                # ---- moving GAP label between Gp and 1/Gc (loop gain / phase margin) ----
                if pid == 'olm':
                    fa = H[-1]['f']; lf = np.log10(fa)
                    lg = H[-1]['Gp_mag'] - H[-1]['invGc_mag']    # loop gain |Gp*Gc| in dB
                    ya = self._interp((fa, H[-1]['Gp_mag']), x, True)
                    yb = self._interp((fa, H[-1]['invGc_mag']), x, True)
                    gapv = self._interp((fa, lg), x, True)
                    j = int(np.argmin(np.abs(lf - np.log10(x))))
                    k = max(1, len(lf)//120)
                    a = max(0, j-k); b = min(len(lf)-1, j+k)
                    slope = (lg[b]-lg[a])/(lf[b]-lf[a]) if lf[b] != lf[a] else 0.0
                    arts.append(ax.annotate('', xy=(x, ya), xytext=(x, yb),
                                arrowprops=dict(arrowstyle='<->', color='#1565c0', lw=1.4,
                                                alpha=0.9), zorder=17))
                    arts.append(ax.annotate(f"$G_p\\!\\cdot\\!G_c$ = {gapv:+.1f} dB, {slope:+.0f} dB/dec",
                                xy=(x, 0.5*(ya+yb)), xytext=(8, -7), textcoords='offset points',
                                va='center', fontsize=8, color='#1565c0', fontweight='bold',
                                bbox=dict(boxstyle='round,pad=0.25', fc='#eaf2fb',
                                          ec='#1565c0', alpha=0.95), zorder=18))
                elif pid == 'olp':
                    ya = self._interp((H[-1]['f'], H[-1]['Gp_ph']), x, True)
                    yb = self._interp((H[-1]['f'], H[-1]['invGc_ph_s']), x, True)
                    gapv = ya - yb                      # phase gap (= PM at crossover)
                    arts.append(ax.annotate('', xy=(x, ya), xytext=(x, yb),
                                arrowprops=dict(arrowstyle='<->', color='#1565c0', lw=1.4,
                                                alpha=0.9), zorder=17))
                    arts.append(ax.annotate(f"$G_p\\!\\cdot\\!G_c$ = {gapv:+.0f}°",
                                xy=(x, 0.5*(ya+yb)), xytext=(8, -7), textcoords='offset points',
                                va='center', fontsize=8, color='#1565c0', fontweight='bold',
                                bbox=dict(boxstyle='round,pad=0.25', fc='#eaf2fb',
                                          ec='#1565c0', alpha=0.95), zorder=18))
            self._cursors[pid] = arts
        self.fig.canvas.draw_idle()

    def _place_sticky_label(self, ax, pid, xy, text, color):
        """Annotate avoiding overlap with this panel's existing sticky labels."""
        lab = ax.annotate(text, xy=xy, xytext=(10, 12), textcoords='offset points',
                          fontsize=8, fontweight='bold', color=color, zorder=17,
                          bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, alpha=0.95))
        try:
            rend = self.fig.canvas.get_renderer()
            existing = [g['label'].get_window_extent(rend) for g in self._sticky[pid]]
            for off in [(10, 12), (10, -34), (10, 56), (10, -78), (-95, 12), (-95, -34)]:
                lab.xyann = off
                if not any(lab.get_window_extent(rend).overlaps(e) for e in existing):
                    break
        except Exception:
            pass
        return lab

    def _on_click(self, ev):
        if ev.inaxes not in self._ax2id or ev.xdata is None:
            return
        pid = self._ax2id[ev.inaxes]; ax = self.ax[pid]
        AZURE = '#1e90ff'
        # age existing stickies: solid(0) -> dashed(1) -> removed(>=2)
        survivors = []
        for grp in self._sticky[pid]:
            grp['age'] += 1
            if grp['age'] >= 2:
                for a in grp['artists']:
                    try: a.remove()
                    except Exception: pass
            else:
                grp['line'].set_linestyle('--'); grp['line'].set_alpha(0.7)
                grp['label'].set_alpha(0.7)
                survivors.append(grp)
        self._sticky[pid] = survivors
        if pid in self.TIME:
            cur = self._curves.get(pid, [])
            yi = self._interp(cur[-1], ev.xdata, False) if cur else ev.ydata
        else:
            yi = ev.ydata
        vline = ax.axvline(ev.xdata, color=AZURE, lw=1.4, zorder=16)
        lab = self._place_sticky_label(ax, pid, (ev.xdata, yi),
                                       self._fmt(pid, ev.xdata, yi), AZURE)
        self._sticky[pid].append(dict(artists=[vline, lab], line=vline, label=lab, age=0))
        self.fig.canvas.draw_idle()

## Interactive controls
Drag a slider **or type a value**. Units: $V_{in}$/$V_{out}$ in V, $L$ in µH, $C$ in µF,
$r_C$/$r_L$ in mΩ, $f_{sw}$/$f_{c,t}$ in kHz, $R_{load}$ in Ω, $R_i$ in V/A, $S_e$ in V/µs. Each
slider is centred on its AN-1162 value and spans **0.01× to 100×** (ticks at 0.01×, 0.1×, 1×, 10×,
100×; $S_e$ is linear 0–6.8 V/µs). Try: raise $R_{load}$ past ~2.1 Ω to watch **auto** flip into
DCM; switch **control** to *current* and push $V_{out}$ above 6 V with $S_e=0$ to trip the
**subharmonic** warning, then restore it with $S_e$. Plot traces are **black** (previous curves
dashed/dotted). Hover any plot for a live readout mirrored into sibling plots; click to drop a
sticky marker. **Reset** clears everything.

In [ ]:
import numpy as np
import ipywidgets as widgets
from ipywidgets import (FloatLogSlider, FloatSlider, FloatText, Button,
                        ToggleButtons, Dropdown, HBox, VBox, Label, Layout, link)
from IPython.display import display

# slider CENTERS in display units (AN-1162 Type III-B design [1]):
#   Vin[V] Vout[V] L[µH] C[µF] rC[mΩ] rL[mΩ] fsw[kHz] Rload[Ω] Ri[V/A] Se[V/µs] fct[kHz]
Vin0, Vout0, L0, C0 = 12.0, 1.8, 1.5, 43.2
rC0, rL0, fsw0, R0  = 0.75, 5.0, 600.0, 0.45
Ri0, Se0, fct0      = 0.10, 0.68, 100.0

def to_P():
    """Convert current widget values (display units) to SI for the physics."""
    return dict(Vin=Vin_s.value,            # V
                Vout=Vout_s.value,          # V
                L=L_s.value*1e-6,           # µH  -> H
                C=C_s.value*1e-6,           # µF  -> F
                rC=rC_s.value*1e-3,         # mΩ  -> Ω
                rL=rL_s.value*1e-3,         # mΩ  -> Ω
                fsw=fsw_s.value*1e3,        # kHz -> Hz
                Rload=R_s.value,            # Ω
                Ri=Ri_s.value,              # V/A
                Se=Se_s.value*1e6,          # V/µs -> V/s
                fct=fct_s.value*1e3,        # kHz -> Hz
                mode=mode_t.value, control=ctrl_t.value, comp=comp_d.value)

explorer = BuckExplorer()

# log sliders span exactly 0.01x .. 100x of centre (ticks at 0.01x, 0.1x, 1x, 10x, 100x);
# Se is linear (0 must be reachable: no slope compensation).
def log_slider(center):
    e = np.log10(center)
    return FloatLogSlider(value=center, base=10, min=e-2, max=e+2, step=0.01,
                          readout=False, continuous_update=False,
                          layout=Layout(width='260px'))

def lin_slider(center, lo, hi, step):
    return FloatSlider(value=center, min=lo, max=hi, step=step,
                       readout=False, continuous_update=False,
                       layout=Layout(width='260px'))

Vin_s  = log_slider(Vin0)
Vout_s = log_slider(Vout0)
L_s    = log_slider(L0)
C_s    = log_slider(C0)
rC_s   = log_slider(rC0)
rL_s   = log_slider(rL0)
fsw_s  = log_slider(fsw0)
R_s    = log_slider(R0)
Ri_s   = log_slider(Ri0)
Se_s   = lin_slider(Se0, 0.0, 6.8, 0.01)     # V/µs; 0 = no slope comp, 10x Sn0 max
fct_s  = log_slider(fct0)
allw = (Vin_s, Vout_s, L_s, C_s, rC_s, rL_s, fsw_s, R_s, Ri_s, Se_s, fct_s)

ROW_COMP = {'Vin (V)': 'Vin', 'Vout (V)': 'R', 'L (µH)': 'L', 'C (µF)': 'C',
            'rC (mΩ)': 'C', 'rL (mΩ)': 'L', 'fsw (kHz)': 'SW', 'Rload (Ω)': 'R',
            'Ri (V/A)': 'SW', 'Se (V/µs)': 'SW', 'fc,t (kHz)': 'SW'}
_ui = {'reset': False}                       # suppress auto-highlight during reset

_CENTERS = {'Vin (V)': Vin0, 'Vout (V)': Vout0, 'L (µH)': L0, 'C (µF)': C0,
            'rC (mΩ)': rC0, 'rL (mΩ)': rL0, 'fsw (kHz)': fsw0, 'Rload (Ω)': R0,
            'Ri (V/A)': Ri0, 'Se (V/µs)': Se0, 'fc,t (kHz)': fct0}
# half-decade multipliers of the slider's ORIGINAL value (spans the full 0.01x..100x range)
_MULT = [0.01, 0.032, 0.1, 0.32, 1.0, 3.2, 10.0, 32.0, 100.0]
def _mult_next(v, center, up):
    if center <= 0 or v <= 0:
        return v
    m = v/center; eps = 1e-3
    if up:
        pick = [mm for mm in _MULT if mm > m*(1+eps)]
        return center*(pick[0] if pick else _MULT[-1])
    pick = [mm for mm in _MULT if mm < m*(1-eps)]
    return center*(pick[-1] if pick else _MULT[0])

def _step(s, kind, up, center=None):
    if kind == 'lin':                        # Se: linear V/µs, +/- 0.34 (half of Sn0)
        nv = min(s.max, max(s.min, s.value + (0.34 if up else -0.34)))
    else:                                    # log sliders: x-of-original half-decade ladder
        lo, hi = 10.0**s.min, 10.0**s.max
        nv = min(hi, max(lo, _mult_next(s.value, center, up)))
    s.value = round(nv, 6)

def row(desc, s):
    comp = ROW_COMP[desc]
    kind = 'lin' if desc == 'Se (V/µs)' else 'mult'
    center = _CENTERS[desc]
    t = FloatText(value=s.value, layout=Layout(width='90px'))
    link((s, 'value'), (t, 'value'))
    lbl = Button(description=desc, tooltip=f'highlight {comp} (click again to clear)',
                 layout=Layout(width='95px'),
                 style={'button_color': 'transparent', 'font_weight': 'normal'})
    lbl.on_click(lambda _b, c=comp: explorer.set_color(None if explorer.color_state == c else c))
    s.observe(lambda _ch, c=comp: (None if _ui['reset'] else explorer.set_color(c)),
              names='value')
    tip = ('-0.34 V/µs', '+0.34 V/µs') if kind == 'lin' else \
          ('step down (x of orig.)', 'step up (x of orig.)')
    dn = Button(description='\u25bc', layout=Layout(width='32px'), tooltip=tip[0])
    up = Button(description='\u25b2', layout=Layout(width='32px'), tooltip=tip[1])
    dn.on_click(lambda _b, ss=s, k=kind, c=center: _step(ss, k, False, c))
    up.on_click(lambda _b, ss=s, k=kind, c=center: _step(ss, k, True, c))
    return HBox([lbl, s, t, dn, up])

rows = VBox([row('Vin (V)',    Vin_s),
             row('Vout (V)',   Vout_s),
             row('L (µH)',     L_s),
             row('C (µF)',     C_s),
             row('rC (mΩ)',    rC_s),
             row('rL (mΩ)',    rL_s),
             row('fsw (kHz)',  fsw_s),
             row('Rload (Ω)',  R_s),
             row('Ri (V/A)',   Ri_s),
             row('Se (V/µs)',  Se_s),
             row('fc,t (kHz)', fct_s)])

mode_t = ToggleButtons(options=['auto', 'CCM', 'DCM'], value='auto',
                       description='mode', style={'button_width': '62px'})
ctrl_t = ToggleButtons(options=['voltage', 'current'], value='voltage',
                       description='control', style={'button_width': '82px'})
comp_d = Dropdown(options=['auto', 'type2', 'type3'], value='auto',
                  description='comp', layout=Layout(width='180px'))

reset_btn = Button(description='Reset', button_style='warning',
                   icon='undo', layout=Layout(width='120px'))
color_btn = Button(description='Color', icon='paint-brush',
                   layout=Layout(width='120px'))

def redraw(*_):
    explorer.update(to_P())

for w in allw:
    w.observe(redraw, names='value')
for w in (mode_t, ctrl_t, comp_d):
    w.observe(redraw, names='value')

# Color button cycles: off -> ALL -> Vin -> SW -> L -> C -> R -> ALL -> ...  Reset clears.
COLOR_CYCLE = ['ALL', 'Vin', 'SW', 'L', 'C', 'R']
def on_color(_):
    st = explorer.color_state
    idx = 0 if st is None else (COLOR_CYCLE.index(st) + 1) % len(COLOR_CYCLE)
    explorer.set_color(COLOR_CYCLE[idx])
color_btn.on_click(on_color)

def on_reset(_):
    _ui['reset'] = True
    for w in allw: w.unobserve(redraw, names='value')
    for w in (mode_t, ctrl_t, comp_d): w.unobserve(redraw, names='value')
    explorer.reset()
    (Vin_s.value, Vout_s.value, L_s.value, C_s.value, rC_s.value, rL_s.value,
     fsw_s.value, R_s.value, Ri_s.value, Se_s.value, fct_s.value) = \
        (Vin0, Vout0, L0, C0, rC0, rL0, fsw0, R0, Ri0, Se0, fct0)
    mode_t.value, ctrl_t.value, comp_d.value = 'auto', 'voltage', 'auto'
    for w in allw: w.observe(redraw, names='value')
    for w in (mode_t, ctrl_t, comp_d): w.observe(redraw, names='value')
    _ui['reset'] = False
    explorer.set_color(None)
    redraw()

reset_btn.on_click(on_reset)

# LEFT column = schematic over sliders/toggles/buttons/warnings; formula image on the RIGHT.
left_col = VBox([explorer.sch_widget, rows,
                 HBox([mode_t]), HBox([ctrl_t, comp_d]),
                 HBox([reset_btn, color_btn]),
                 explorer.warn_widget])
top_row = HBox([left_col, explorer.eq_img], layout=Layout(margin='10px 0 0 0'))
display(top_row)
display(explorer.fig.canvas)
redraw()

---
## Parameter validation (single source: AN-1162)

All default parameters are taken from one reputable, widely-cited reference — International
Rectifier (now Infineon), *AN-1162: Compensator Design Procedure for Buck Converter with
Voltage-Mode Error-Amplifier* [**[1]**](#References) — the **Type III-B design example** (IR3842,
12 V → 1.8 V @ 4 A, 600 kHz, MLCC output bank of 4×22 µF derated to 4×10.8 µF effective). The one
exception is $r_L$ = 5 mΩ (assumed typical DCR — [1] models $R_L$ symbolically but the example
gives no value); the current-mode defaults $R_i$, $S_e$ come from the supporting sources [3][4].

Running this notebook's model with those values reproduces AN-1162's published numbers:

| Quantity | AN-1162 [1] | This model | Note |
|---|---|---|---|
| $F_{LC}$ (double pole) | 19.7 kHz | 19.9 kHz | model folds $r_L$, $r_C$ into $\omega_0$ |
| $F_{ESR}$ | 4.9 MHz | 4.91 MHz | $r_C=0.75$ mΩ, $C=43.2$ µF |
| Comp placement $F_{z1},F_{z2},F_{p2},F_{p3}$ | 8.8 k, 17.6 k, 567 k, 300 kHz | 8.8 k, 17.6 k, 567 k, 300 kHz | auto Type III-B, $\theta=70°$ lead pair |
| Loop with **exact standard R/C values** ($R_{f1}$=4.02 k, $R_{f3}$=127, $C_{f3}$=2.2 n, $R_{C1}$=2.74 k, $C_{C1}$=6.8 n, $C_{C2}$=180 p) | measured $F_0$=105 kHz, PM≈51° | 98.9 kHz, 55.0° | averaged model; measurement adds modulator delay |
| Loop with auto compensator, $f_{c,t}$=100 kHz | target $F_0$=100 kHz, PM>45° | 100.0 kHz, 52.9°, GM 19.5 dB | |

The agreement across the power-stage corners, the compensator placement, and the closed-loop
crossover/PM validates the plant, compensator, and loop models against the published, measured
design. The DCM small-signal model and the current-mode sampling model are validated structurally
against the supporting sources [2][3][4] (pole locations and the $Q_p$ subharmonic criterion),
which are labeled *supporting*, not primary.

---
## Design questions asked in an interview

**Q1[NVIDIA-Glassdoor].** From a hardware-engineer interview: "Give you the circuit of buck
converter, boost converter and buck-boost converter" — then identify each and explain how it works
[**[7]**](#References).

Identification is by where the inductor and switch sit relative to the input. **Buck** (this
notebook's schematic): switch in series with $V_{in}$, then the LC filter — inductor current is
*drawn from the input only during $DT_s$*, so the output averages down, $V_{out}=D\,V_{in}$ in CCM
(volt-second balance on $L$: $(V_{in}-V_{out})D = V_{out}(1-D)$) [**[10]**](#References).
**Boost**: inductor in series with the input, switch shunting to ground — energy stored in $L$ is
released above $V_{in}$, $V_{out}=V_{in}/(1-D)$. **Buck-boost**: switch first, then a shunt
inductor — output inverted, $V_{out}=-V_{in}\,D/(1-D)$. The follow-up an interviewer usually wants:
the buck's control-to-output transfer function has **no right-half-plane zero**, while boost and
buck-boost in CCM do — which is why the buck loop (this explorer) is the "clean" compensation case
and the boost is not [**[2]**](#References). Use the schematic and the $D$, $M$ ledger rows to
anchor the explanation.

**Q2[Apple-Taro].** From an Apple interview-prep bank on buck design considerations, including
modes of operation and passives selection: "How do you determine the appropriate inductance value
to ensure CCM operation" (with follow-ups on capacitor sizing for a ripple spec)
[**[8]**](#References).

The boundary is exactly this notebook's $K$ row: $K = 2Lf_{sw}/R_{load}$ vs $K_{crit}=1-M$ — CCM
requires $K>1-M$, i.e. $L > (1-M)R_{load}/(2f_{sw})$ at the lightest load you must hold in CCM
[**[2]**](#References). Practically you size $L$ from the ripple guideline
$\Delta I_L \approx 40\%\,I_{o,max}$: $L = \frac{V_{in}-V_{out}}{\Delta I_L}\cdot
\frac{V_{out}}{V_{in}}\cdot\frac{1}{f_{sw}}$ (AN-1162 eq. A2 — the defaults' 1.5 µH gives 42 %)
[**[1]**](#References). The capacitor then comes from the ripple *and* the transient budget:
$\Delta V_{out}\approx\Delta I_L\,(r_C + 1/(8Cf_{sw}))$ for switching ripple, plus
$C_{o,min}= L\,I_{step}^2/(2V_{out}\Delta V_{max})$ for load steps (AN-1162 eq. A3). CCM is
preferred at full load for lower peak/RMS current and ripple; DCM appears at light load and changes
the plant from 2nd- to 1st-order [**[6]**](#References) — flip the **mode** toggle (or just raise
$R_{load}$) and watch rows 2–4 change together.

**Q3[Qualcomm-Glassdoor].** From a Qualcomm PMIC design-engineer interview report — converter
waveforms plus "boost converter waveform, compensation" and control questions
[**[9]**](#References). The canonical form: *compensate a buck's feedback loop in voltage mode vs
current mode; when do you need slope compensation?*

Voltage-mode CCM: the plant is the LC double pole ($-40$ dB/dec, up to $-180°$ phase) plus the ESR
zero, so a **Type III** compensator (integrator + two zeros + two HF poles) supplies the phase
boost; place the crossover at $f_{sw}/10$–$f_{sw}/5$ with PM > 45° (AN-1162's rules, used verbatim
by this notebook's auto compensator) [**[1]**](#References). Peak current mode: the inner current
loop absorbs the inductor state, leaving a **single load pole** — a Type II compensator suffices
[**[4]**](#References) — but sampling adds a double pole at $f_{sw}/2$ with
$Q_p = 1/(\pi(m_c(1-D)-\tfrac12))$: for $D>0.5$ with no external ramp, $Q_p$ goes negative and the
current loop breaks into period-doubling **subharmonic oscillation**; an added ramp $S_e$
(typically $S_e \approx S_f$, i.e. $m_c\approx 2$) restores it [**[3]**](#References)
[**[5]**](#References). Demonstrate live: control → *current*, $V_{out}$ → 7.2 V ($D=0.6$),
$S_e=0$ → warning + peaked $|T|$ at $f_{sw}/2$ + diverging load step; slide $S_e$ up to watch
$Q_p$ in the ledger come back positive.

---
*Disclaimer: this notebook contains AI-generated content. Formulas, derivations, sourced
quotations, and interview answers were checked against the cited references at build time, but may
contain errors and are provided "as is," without warranty of any kind, for educational purposes
only. Interview questions are quoted from public web sources and belong to their respective
posters/platforms; company names identify the reported interview context and imply no affiliation
or endorsement. Verify against the primary sources before using in a real design or interview.*

---
## References

[1] <a id="ref1"></a>A. M. Rahimi, P. Parto, and P. Asadi, *AN-1162: Compensator Design Procedure
for Buck Converter with Voltage-Mode Error-Amplifier*, International Rectifier (now Infineon)
application note.
https://www.infineon.com/assets/row/public/documents/24/42/Infineon-an-1162-ApplicationNotes-vNA-EN.pdf
— **primary source of every default design parameter** ($V_{in}$, $V_{out}$, $I_o$, $L$, $C_o$,
ESR, $f_{sw}$, $V_{osc}$, $F_0$) and of the compensator placement rules and validation targets.
Its Type III-B example reports the loop measurement quoted here: "a crossover frequency of
F0 = 105kHz and the phase-margin of about 51º."

[2] R. W. Erickson and D. Maksimovic, *Fundamentals of Power Electronics*, 2nd ed., Springer,
ISBN 978-0-7923-7270-7 — CCM/DCM conversion ratios and DCM small-signal model; lead-compensator
placement (also cited as ref. [3] inside AN-1162). *Supporting source.*

[3] R. B. Ridley, "A New, Continuous-Time Model for Current-Mode Control," *IEEE Transactions on
Power Electronics*, vol. 6, no. 2, pp. 271–280, 1991 — the CM-CCM plant used here: load pole,
$K_x$, and the $f_{sw}/2$ sampling double pole with $Q_p=1/(\pi(m_c D'-\tfrac12))$.
*Supporting source.*

[4] Richtek, *AN028: Peak-Current Mode Buck Converter Compensation Design*.
https://www.richtek.com/~/media/AN%20PDF/AN028_EN.pdf — Type II compensation of the peak
current-mode buck (zero cancels the low-frequency load pole; HF poles at $f_{sw}/2$).
*Supporting source.*

[5] S. Franco, *Slope Compensation in PCMC DC-DC Converters*, EDN.
https://www.edn.com/slope-compensation-in-pcmc-dc-dc-converters/ — mechanism of peak-current-mode
subharmonic oscillation and its removal by an added ramp. *Supporting source (Q3).*

[6] Monolithic Power Systems, *The Difference between CCM and DCM Explained*.
https://media.monolithicpower.com/mps_cms_document/2/0/2021-seo-the-difference-between-ccm-and-dcm-explained_r1.0.pdf
— CCM vs DCM tradeoffs (ripple, efficiency, light load). *Supporting source (Q2).*

[7] Glassdoor, NVIDIA hardware-engineer interview question: "Give you the circuit of buck
converter, boost converter and buck-boost converter."
https://www.glassdoor.co.uk/Interview/1-Give-you-the-circuit-of-buck-converter-boost-converter-and-buck-boost-converter-Ask-you-to-identify-them-and-explain-QTN_6337896.htm
*(Q1 source.)*

[8] Taro (jointaro.com), Apple interview insights: *Explain the design considerations for a DC-DC
buck converter* — asks "How do you determine the appropriate inductance value to ensure CCM
operation."
https://www.jointaro.com/interview-insights/apple/explain-the-design-considerations-for-a-dc-dc-buck-converter/
*(Q2 source.)*

[9] Glassdoor, Qualcomm Power Management IC Design Engineer interview experience — reports circuit
questions including "boost converter waveform, compensation."
https://www.glassdoor.com/Interview/Qualcomm-Power-Management-IC-Design-Engineer-Interview-Questions-EI_IE640.0,8_KO9,44.htm
*(Q3 source.)*

[10] ECExplain, *DC-DC Converter Interview Questions* — volt-second-balance derivations of the
buck/boost conversion ratios and the CCM/DCM comparison.
https://ecexplain.in/interview/dc-dc-converter-interview *(supporting, Q1.)*